In [1]:
import pandas as pd
import re
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score, accuracy_score, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE ATRIBUTOS Y ARCHIVOS ---

# 1. Lista final de los 35 descriptores de consenso (J48 ∩ IBk)
ATRIBUTOS_DE_CONSENSO = [
    'nBonds', 'nBondsD', 'F04[C-C]', 'F09[C-O]', 'nX', 'piPC05', 'TpiPC', 
    'AATS5i', 'AATS6i', 'GATS5i', 'BCUTp-1l', 'Eig12_AEA(dm)', 
    'MACCSFP159', 'SsssCH', 'B07[C-C]', 'SaaCH', 'B05[C-N]', 'B06[C-N]', 
    'B07[N-O]', 'F02[C-N]', 'F03[C-O]', 'F08[C-N]', 'MACCSFP22', 
    'MACCSFP42', 'MACCSFP88', 'MACCSFP102', 'MACCSFP121', 'MACCSFP125', 
    'MACCSFP131', 'MACCSFP141', 'MACCSFP153', 'MACCSFP158', 'MACCSFP161', 
    'MACCSFP163', 'MACCSFP164'
]

# 2. Rutas a los archivos de datos completos
ruta_entrenamiento_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/combined_training.csv"
ruta_prueba_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/fusion_test.csv"
ruta_externa_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/fusion_external.csv"

print(f"✅ Celda 1: Librerías y {len(ATRIBUTOS_DE_CONSENSO)} atributos cargados.")

✅ Celda 1: Librerías y 35 atributos cargados.


In [4]:
print("--- Creando Datasets Finales con los 35 atributos de consenso... ---")

# 1. Cargar las cabeceras del archivo de entrenamiento para ver los nombres disponibles
df_headers = pd.read_csv(ruta_entrenamiento_completo, sep=',', nrows=0)
nombres_disponibles = df_headers.columns.tolist()
columna_clase = nombres_disponibles[-1]

# 2. Comparar tu lista de 35 atributos con los que realmente existen en el archivo
atributos_validos = []
atributos_no_encontrados = []

for atributo_deseado in ATRIBUTOS_DE_CONSENSO:
    if atributo_deseado in nombres_disponibles:
        atributos_validos.append(atributo_deseado)
    else:
        atributos_no_encontrados.append(atributo_deseado)

print(f"> Se encontraron {len(atributos_validos)} de tus {len(ATRIBUTOS_DE_CONSENSO)} atributos en el archivo de entrenamiento.")
if atributos_no_encontrados:
    print(f"  > AVISO: Los siguientes {len(atributos_no_encontrados)} atributos no fueron encontrados y serán ignorados:")
    print(f"    {atributos_no_encontrados}")

# 3. Crear los datasets finales usando solo los atributos válidos
columnas_finales = sorted(atributos_validos) + [columna_clase]

df_train = pd.read_csv(ruta_entrenamiento_completo)[columnas_finales]
df_train.to_csv(r"C:\Users\benja\Desktop\BD PAMPA\Calculos BD reducida/consenso_train.csv", index=False)

df_test = pd.read_csv(ruta_prueba_completo)[columnas_finales]
df_test.to_csv(r"C:\Users\benja\Desktop\BD PAMPA\Calculos BD reducida/consenso_test.csv", index=False)

df_external = pd.read_csv(ruta_externa_completo)[columnas_finales]
df_external.to_csv(r"C:\Users\benja\Desktop\BD PAMPA\Calculos BD reducida/consenso_external.csv", index=False)

print("\n  > Archivos 'consenso_final_train.csv', etc., creados con los atributos válidos.")

# 4. Preparamos los datos para las siguientes celdas
X_train, y_train = df_train.drop(columns=[columna_clase]), df_train[columna_clase]
X_test, y_test = df_test.drop(columns=[columna_clase]), df_test[columna_clase]
X_external, y_external = df_external.drop(columns=[columna_clase]), df_external[columna_clase]

print("\n✅ Celda 2: Datos cargados y listos para el modelado.")

--- Creando Datasets Finales con los 35 atributos de consenso... ---
> Se encontraron 24 de tus 35 atributos en el archivo de entrenamiento.
  > AVISO: Los siguientes 11 atributos no fueron encontrados y serán ignorados:
    ['nBonds', 'nBondsD', 'F09[C-O]', 'TpiPC', 'AATS5i', 'AATS6i', 'BCUTp-1l', 'B05[C-N]', 'F02[C-N]', 'F03[C-O]', 'F08[C-N]']

  > Archivos 'consenso_final_train.csv', etc., creados con los atributos válidos.

✅ Celda 2: Datos cargados y listos para el modelado.


In [6]:
print("--- Iniciando Evaluación Comparativa de Modelos... ---")

modelos_a_evaluar = {
    "SVM": SVC(probability=True, random_state=42), 
    "Random Forest": RandomForestClassifier(random_state=42),
    "Árbol de Decisión (J48)": DecisionTreeClassifier(random_state=42),
    "k-NN (IBk)": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

resultados_comparativos = {}
for nombre_modelo, modelo in modelos_a_evaluar.items():
    modelo.fit(X_train, y_train)
    roc_auc_test = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
    resultados_comparativos[nombre_modelo] = roc_auc_test
    print(f"  > {nombre_modelo:<25} | ROC AUC (Test): {roc_auc_test:.4f}")

campeon_nombre = max(resultados_comparativos, key=resultados_comparativos.get)
print(f"\n✅ Celda 3: El modelo campeón en la comparativa es: {campeon_nombre}")

--- Iniciando Evaluación Comparativa de Modelos... ---
  > SVM                       | ROC AUC (Test): 0.7765
  > Random Forest             | ROC AUC (Test): 0.8067
  > Árbol de Decisión (J48)   | ROC AUC (Test): 0.6589
  > k-NN (IBk)                | ROC AUC (Test): 0.7348
  > Naive Bayes               | ROC AUC (Test): 0.7463

✅ Celda 3: El modelo campeón en la comparativa es: Random Forest


In [8]:
print(f"--- Optimizando al Campeón ({campeon_nombre})... ---")

# Grids de parámetros para los modelos más prometedores
param_grids = {
    "Random Forest": {'n_estimators': [100, 200, 300], 'max_features': ['sqrt', 'log2']},
    "SVM": {'C': [1, 10, 100], 'gamma': ['scale', 'auto']}
}

if campeon_nombre in param_grids:
    grid_search = GridSearchCV(modelos_a_evaluar[campeon_nombre], param_grids[campeon_nombre], cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train)
    modelo_final = grid_search.best_estimator_
    print(f"\n> Mejores parámetros encontrados: {grid_search.best_params_}")
else:
    print(f"\n> No hay grid de optimización definido para {campeon_nombre}. Se usará el modelo base.")
    modelo_final = modelos_a_evaluar[campeon_nombre]

# --- REPORTE FINAL (CON GUARDADO EN ARCHIVO) ---
nombre_archivo_reporte = "REPORTE_FINAL.txt"
with open(nombre_archivo_reporte, 'w') as f:
    
    # --- Escribiendo y mostrando el encabezado ---
    encabezado = "="*60 + "\n--- REPORTE FINAL DEL MODELO DE CONSENSO ---\n" + "="*60
    print("\n" + encabezado)
    f.write(encabezado + "\n\n")
    
    # --- Bucle para escribir y mostrar los resultados ---
    for nombre_set, X_eval, y_eval in [("Prueba Interna (Test Set)", X_test, y_test), ("Prueba Externa (External Set)", X_external, y_external)]:
        y_pred = modelo_final.predict(X_eval)
        y_proba = modelo_final.predict_proba(X_eval)[:, 1]
        
        # Generamos el bloque de texto para este conjunto de datos
        bloque_resultados = (
            f"Resultados en: {nombre_set}\n"
            f"----------------------------------------\n"
            f"  Accuracy:         {accuracy_score(y_eval, y_pred):.4f}\n"
            f"  ROC AUC:          {roc_auc_score(y_eval, y_proba):.4f}\n"
            f"  BACC:             {balanced_accuracy_score(y_eval, y_pred):.4f}\n"
            f"  Sensitivity:      {recall_score(y_eval, y_pred, pos_label='Act1'):.4f}\n"
            f"  Specificity:      {recall_score(y_eval, y_pred, pos_label='Act-1'):.4f}\n"
            f"  Kappa:            {cohen_kappa_score(y_eval, y_pred):.4f}\n"
        )
        
        # Lo mostramos en pantalla
        print("\n" + bloque_resultados)
        # Y lo escribimos en el archivo
        f.write(bloque_resultados + "\n")

print(f"\n\n✅ ¡PROYECTO COMPLETADO! Resultados guardados en '{nombre_archivo_reporte}'.")

--- Optimizando al Campeón (Random Forest)... ---
Fitting 5 folds for each of 6 candidates, totalling 30 fits

> Mejores parámetros encontrados: {'max_features': 'sqrt', 'n_estimators': 300}

--- REPORTE FINAL DEL MODELO DE CONSENSO ---

Resultados en: Prueba Interna (Test Set)
----------------------------------------
  Accuracy:         0.7367
  ROC AUC:          0.8068
  BACC:             0.7337
  Sensitivity:      0.7770
  Specificity:      0.6903
  Kappa:            0.4690


Resultados en: Prueba Externa (External Set)
----------------------------------------
  Accuracy:         0.8663
  ROC AUC:          0.6235
  BACC:             0.6012
  Sensitivity:      0.9233
  Specificity:      0.2791
  Kappa:            0.1961



✅ ¡PROYECTO COMPLETADO! Resultados guardados en 'REPORTE_FINAL.txt'.
